In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt


In [ ]:
url = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
df = pd.read_csv(url)

# Keep only the required columns
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[cols]

print("Dataset shape:", df.shape)
df.head()

In [ ]:
df['fuel_efficiency_mpg'].hist(bins=30)
plt.title("Fuel Efficiency Distribution")
plt.xlabel("MPG")
plt.ylabel("Frequency")
plt.show()

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Question 1 answer: Column with missing values
col_missing = df.isnull().sum().idxmax()
print(f"\nQ1: Column with missing values is: '{col_missing}'")

In [ ]:
# Question 2: Median of horsepower

median_hp = df['horsepower'].median()
print(f"\nQ2: Median horsepower = {median_hp}")

In [ ]:
def split_data(data, seed=42):
    df_shuffled = data.sample(frac=1, random_state=seed)
    n = len(df_shuffled)
    n_train = int(0.6 * n)
    n_val = int(0.2 * n)
    
    df_train = df_shuffled.iloc[:n_train]
    df_val = df_shuffled.iloc[n_train:n_train + n_val]
    df_test = df_shuffled.iloc[n_train + n_val:]
    
    return df_train, df_val, df_test

df_train, df_val, df_test = split_data(df, seed=42)

# Separate features and target
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

X_train = df_train[features].copy()
y_train = df_train['fuel_efficiency_mpg'].copy()

X_val = df_val[features].copy()
y_val = df_val['fuel_efficiency_mpg'].copy()

X_test = df_test[features].copy()
y_test = df_test['fuel_efficiency_mpg'].copy()

In [ ]:
# Question 3: Missing value strategies

# Option 1: Fill with 0
X_train_0 = X_train.fillna(0)
X_val_0 = X_val.fillna(0)

In [ ]:
# Option 2: Fill with mean (based on training data)
mean_hp = X_train['horsepower'].mean()
X_train_mean = X_train.fillna({'horsepower': mean_hp})
X_val_mean = X_val.fillna({'horsepower': mean_hp})

In [ ]:
# Train Linear Regression (no regularization)
lr = LinearRegression()

In [ ]:
# Train + Evaluate (fill=0)
lr.fit(X_train_0, y_train)
y_pred_0 = lr.predict(X_val_0)
rmse_0 = np.sqrt(mean_squared_error(y_val, y_pred_0))

In [ ]:
# Train + Evaluate (fill=mean)
lr.fit(X_train_mean, y_train)
y_pred_mean = lr.predict(X_val_mean)
rmse_mean = np.sqrt(mean_squared_error(y_val, y_pred_mean))

In [ ]:
print(f"\nQ3: RMSE (fill 0)   = {round(rmse_0, 2)}")
print(f"Q3: RMSE (fill mean) = {round(rmse_mean, 2)}")

In [ ]:
if rmse_mean < rmse_0:
    better = "mean"
else:
    better = "0"
print(f"Better option: fill with {better}")

In [ ]:
# Question 4: Regularized Regression (Ridge)

X_train_0 = X_train.fillna(0)
X_val_0 = X_val.fillna(0)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    model = Ridge(alpha=r)
    model.fit(X_train_0, y_train)
    y_pred = model.predict(X_val_0)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f"r={r:<6} -> RMSE={round(rmse,2)}")

In [ ]:
# Question 5: Different random seeds
rmses = []

for seed in range(10):
    df_train, df_val, df_test = split_data(df, seed=seed)

    X_train = df_train[features].fillna(0)
    y_train = df_train['fuel_efficiency_mpg']

    X_val = df_val[features].fillna(0)
    y_val = df_val['fuel_efficiency_mpg']

    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    rmses.append(rmse)

std_rmse = np.std(rmses)
print(f"\nQ5: Standard deviation of RMSEs = {round(std_rmse, 3)}")

In [ ]:
# Question 6: Train with seed=9, r=0.001
# ----------------------------
df_train, df_val, df_test = split_data(df, seed=9)

df_full_train = pd.concat([df_train, df_val])
X_full_train = df_full_train[features].fillna(0)
y_full_train = df_full_train['fuel_efficiency_mpg']

X_test = df_test[features].fillna(0)
y_test = df_test['fuel_efficiency_mpg']

ridge = Ridge(alpha=0.001)
ridge.fit(X_full_train, y_full_train)
y_pred_test = ridge.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"\nQ6: RMSE on test = {round(rmse_test, 2)}")
